# Methods Tutorial: Fine-Tuning Whisper with Soft Prompts

## Introduction

This tutorial teaches how to adapt **OpenAI Whisper large-v3-turbo** using **soft prompt tuning**.

Soft prompts are trainable continuous vectors. Instead of editing the text prompt manually or updating the full model, we freeze Whisper and learn a small set of prompt embeddings that steer the decoder during training.

Everything needed for the demo is generated inside this notebook. The audio examples are mock 16 kHz waveforms with short transcripts, so no external dataset files are required.

### What You'll Learn

By the end of this tutorial, you will understand:

- What soft prompts are and how they differ from text prompts
- Why prompt tuning can be more lightweight than full fine-tuning
- How to create mock ASR training data
- How to freeze Whisper large-v3-turbo
- How to create trainable decoder prompt embeddings
- How to prepend soft prompts to decoder input embeddings
- How to verify that only soft prompt parameters receive gradients


## Tutorial Layout

| Part | Section | Topic |
|------|---------|-------|
| **Opening** | Introduction | Goals, scope, and what you will learn |
| **Opening** | Overview | Soft prompt tuning pipeline |
| **Background** | A | What are soft prompts? |
| **Background** | B | Soft prompts vs LoRA |
| **Background** | C | Mock ASR training data |
| **Setup** | - | Install from `requirements.txt` |
| **Setup** | 1 | Import libraries |
| **Setup** | 2 | Define helper functions |
| **Setup** | 3 | Create mock data |
| **Implementation** | 4 | Step 1 - Load Whisper large-v3-turbo |
| **Implementation** | 5 | Step 2 - Build trainable soft prompts |
| **Implementation** | 6 | Step 3 - Prepare decoder embeddings |
| **Implementation** | 7 | Step 4 - Run a training smoke test |
| **Visualization** | 8 | Prompt parameter and loss diagnostics |
| **Summary** | 9 | Flow summary and key point |
| **Conclusion** | 10 | What you learned and next steps |


## Overview

Soft prompt tuning follows this pipeline:

1. Create or load paired audio/transcript examples
2. Convert audio into Whisper log-mel features
3. Tokenize transcripts into decoder tokens
4. Freeze all Whisper model weights
5. Create a small trainable prompt embedding matrix
6. Prepend those embeddings to decoder token embeddings
7. Train only the soft prompt vectors

```text
Mock audio + transcript -> processor -> input_features + labels
                                  -> frozen Whisper + trainable soft prompt
                                  -> loss -> optimizer updates prompt only
```

This tutorial uses `openai/whisper-large-v3-turbo` as the target checkpoint. Because that model is large, the real training smoke test is resource-aware: it runs automatically on CUDA and otherwise explains how to enable it.


## A. What Are Soft Prompts?

A normal text prompt is made of discrete tokens, such as:

```text
transcribe this clinical audio
```

A **soft prompt** is different. It is a trainable matrix of continuous vectors:

$$P \in \mathbb{R}^{m \times d}$$

where:

- `P` is the soft prompt matrix
- `m` is the number of prompt tokens
- `d` is the model embedding dimension

The model does not decode these vectors as words. Instead, they are prepended to the decoder input embeddings so they can steer generation through training.


## B. Soft Prompts vs LoRA

| Method | What trains? | Where adaptation lives? | Good for |
|--------|--------------|--------------------------|----------|
| **LoRA** | Low-rank adapter matrices | Inside attention/linear layers | Stronger adaptation with modest trainable parameters |
| **Soft Prompt** | A small prompt embedding table | At the decoder input | Very lightweight task/domain steering |
| **Full Fine-Tuning** | All model weights | Everywhere | Maximum flexibility, highest cost |

Soft prompt tuning usually has fewer trainable parameters than LoRA, but it may also be less expressive.


## C. Mock ASR Training Data

A real ASR fine-tuning dataset contains audio and reference transcripts. This tutorial generates synthetic sine-wave audio and simple mock transcripts.

The mock audio is not meaningful speech. It is used only to verify the training mechanics: tensor shapes, labels, soft prompt insertion, loss computation, and gradient flow.


## Dependencies

Install the required libraries from `requirements.txt`.

This tutorial uses `%pip`, which installs into the active notebook kernel instead of relying on a system-level `pip` command being available on PATH.

The first install line selects the CUDA 13.0 PyTorch wheel, which is appropriate for modern NVIDIA GPUs such as RTX 50-series cards. If a user does not have an NVIDIA GPU, the notebook still falls back to CPU execution, but GPU training requires a CUDA-enabled PyTorch build.


In [ ]:
%pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu130
%pip install -r requirements.txt


## 1. Import Libraries


In [ ]:
# Standard utilities and numerical packages.
import os
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

# PyTorch runs the model and owns the trainable soft prompt parameters.
import torch
import torch.nn as nn

# Transformers loads Whisper and its processor.
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor


## 2. Define Helper Functions

These helpers keep the notebook self-contained and make the soft prompt training steps easier to inspect.


In [ ]:
MODEL_ID = "openai/whisper-large-v3-turbo"
SAMPLE_RATE = 16_000


def choose_device_and_dtype():
    """Select a practical device and dtype for Whisper inference/training."""
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device.startswith("cuda") else torch.float32
    return device, dtype


def should_run_real_training():
    """Run the heavy Whisper smoke test only when explicitly enabled or CUDA is present."""
    flag = os.getenv("RUN_WHISPER_FINETUNE_SMOKE", "").lower()
    return torch.cuda.is_available() or flag in {"1", "true", "yes"}


def create_mock_asr_dataset(num_examples=4, duration_s=1.0, sample_rate=SAMPLE_RATE):
    """Create tiny mock audio/transcript pairs for demonstrating the training pipeline."""
    transcripts = [
        "patient reports blurry vision",
        "left eye pressure is normal",
        "follow up in two weeks",
        "visual acuity improved today",
    ]
    examples = []
    time = np.linspace(0, duration_s, int(sample_rate * duration_s), endpoint=False)
    for idx in range(num_examples):
        freq = 180 + idx * 80
        waveform = 0.15 * np.sin(2 * np.pi * freq * time)
        waveform += 0.02 * np.cos(2 * np.pi * (freq * 1.5) * time)
        waveform = waveform.astype(np.float32)
        examples.append({
            "id": f"mock_{idx:02d}",
            "audio": waveform,
            "sampling_rate": sample_rate,
            "text": transcripts[idx % len(transcripts)],
        })
    return examples


def prepare_audio_inputs(examples, processor, device, dtype):
    """Convert mock audio examples into Whisper input tensors."""
    audio = [ex["audio"] for ex in examples]
    inputs = processor(
        audio,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding="max_length",  # Whisper expects 30-second / 3000-frame mel inputs
        return_attention_mask=True,
    )
    return {k: v.to(device=device, dtype=dtype) for k, v in inputs.items()}


def tokenize_transcripts(examples, processor, device):
    """Tokenize transcripts into decoder inputs and loss labels.

    We need two related tensors:
    - decoder_input_ids: real token IDs used to look up decoder embeddings
    - labels: target token IDs used by the loss function

    Padding tokens should be ignored by the loss, so labels use -100 at padding
    positions. We clone first so decoder_input_ids keep valid token IDs.
    """
    texts = [ex["text"] for ex in examples]
    tokenized = processor.tokenizer(texts, return_tensors="pt", padding=True)

    # These IDs must stay valid because they are passed into the embedding table.
    decoder_input_ids = tokenized.input_ids

    # Labels are a separate copy because the loss uses -100 to ignore padding.
    # If we changed decoder_input_ids in-place, -100 would become an invalid embedding ID.
    labels = decoder_input_ids.clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    return decoder_input_ids.to(device), labels.to(device)


class WhisperSoftPrompt(nn.Module):
    """Trainable prompt vectors that are prepended to decoder token embeddings."""

    def __init__(self, prompt_length, embed_dim, dtype=torch.float32):
        super().__init__()
        prompt = torch.randn(prompt_length, embed_dim, dtype=dtype) * 0.02
        self.prompt_embeddings = nn.Parameter(prompt)

    def forward(self, batch_size):
        return self.prompt_embeddings.unsqueeze(0).expand(batch_size, -1, -1)


def freeze_model(model):
    """Freeze every parameter in the base Whisper model."""
    for param in model.parameters():
        param.requires_grad = False


def count_trainable_parameters(module):
    """Count trainable and total parameters for any torch module."""
    total = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


def plot_waveforms(examples):
    """Visualize the generated mock waveforms."""
    fig, axes = plt.subplots(len(examples), 1, figsize=(10, 1.8 * len(examples)), sharex=True)
    if len(examples) == 1:
        axes = [axes]
    for ax, ex in zip(axes, examples):
        t = np.arange(len(ex["audio"])) / ex["sampling_rate"]
        ax.plot(t, ex["audio"], linewidth=0.8)
        ax.set_title(f"{ex['id']}: {ex['text']}")
        ax.set_ylabel("amp")
    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()
    return fig


def plot_prompt_diagnostics(base_params, prompt_params, losses):
    """Plot parameter counts and optional smoke-test losses."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(["Frozen Whisper", "Soft Prompt"], [base_params, prompt_params], color=["lightgray", "seagreen"])
    axes[0].set_title("Parameter Counts")
    axes[0].set_ylabel("parameters")
    axes[0].ticklabel_format(style="plain", axis="y")

    if losses:
        axes[1].plot(range(1, len(losses) + 1), losses, marker="o")
        axes[1].set_title("Smoke-Test Loss")
        axes[1].set_xlabel("step")
        axes[1].set_ylabel("loss")
    else:
        axes[1].text(0.5, 0.5, "Training smoke test skipped", ha="center", va="center")
        axes[1].set_axis_off()
    plt.tight_layout()
    return fig


## 3. Create Mock Data

The generated dataset is tiny and local to the notebook. It is meant to test the training mechanics, not to improve the model.


In [ ]:
examples = create_mock_asr_dataset(num_examples=4)

print("Mock ASR examples:")
for ex in examples:
    print(f"  {ex['id']} | samples={len(ex['audio']):,} | sr={ex['sampling_rate']} | text='{ex['text']}'")

plot_waveforms(examples)
plt.show()


## 4. Step 1 - Load Whisper large-v3-turbo

The target checkpoint is `openai/whisper-large-v3-turbo`.

The cell is resource-aware. Loading and training this model can be expensive on CPU, so the real model path runs automatically on CUDA or when you set:

```text
RUN_WHISPER_FINETUNE_SMOKE=1
```

This keeps the notebook executable for learners without a GPU while still preserving the exact large-v3-turbo soft prompt tuning code path.


In [ ]:
device, torch_dtype = choose_device_and_dtype()
run_real_training = should_run_real_training()

print(f"Device: {device}")
print(f"Dtype : {torch_dtype}")
print(f"Run real Whisper smoke test: {run_real_training}")

model = None
processor = None

if run_real_training:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    ).to(device)
    freeze_model(model)
    print("Loaded and froze Whisper large-v3-turbo.")
else:
    print("Skipped heavy model loading. Set RUN_WHISPER_FINETUNE_SMOKE=1 or use CUDA to run it.")


## 5. Step 2 - Build Trainable Soft Prompts

The soft prompt is a small trainable matrix with shape:

```text
[prompt_length, decoder_embedding_dim]
```

In the notation from the background section, this is the matrix:

```text
P in R^(m x d)
```

where:

- `m = prompt_length` — the number of learned virtual prompt tokens
- `d = decoder_embedding_dim` — the size of each Whisper decoder embedding vector

The important idea is that **the soft prompt is the only new trainable object**. Whisper is still used to process the audio and compute the loss, but Whisper's original weights are frozen with `requires_grad=False`.

That means training changes only:

```text
soft_prompt.prompt_embeddings
```

and does not change:

```text
Whisper encoder weights
Whisper decoder weights
Whisper token embeddings
Whisper output projection
```

So "Whisper stays frozen" does not mean Whisper is skipped. It means Whisper acts like a fixed backbone. Gradients pass through Whisper so the loss can teach the soft prompt, but the optimizer updates only the prompt matrix `P`.


In [ ]:
prompt_length = 8
losses = []
base_params = 809_000_000
prompt_params = prompt_length * 1280  # Whisper large-v3 family decoder hidden size, used for skipped-path visualization.
soft_prompt = None

if model is not None:
    embed_tokens = model.get_decoder().embed_tokens
    embed_dim = embed_tokens.embedding_dim
    soft_prompt = WhisperSoftPrompt(prompt_length, embed_dim, dtype=torch_dtype).to(device)
    base_params, _ = count_trainable_parameters(model)
    _, prompt_params = count_trainable_parameters(soft_prompt)

    print(f"Decoder embedding dim : {embed_dim}")
    print(f"Prompt length         : {prompt_length}")
    print(f"Soft prompt params    : {prompt_params:,}")
    print("Whisper base params are frozen.")
else:
    print("Soft prompt module creation skipped because the real model was not loaded.")
    print(f"Expected prompt parameters for large-v3-style dim: {prompt_params:,}")


## 6. Step 3 - Prepare Decoder Embeddings

Whisper normally receives decoder token IDs. For soft prompt tuning, we manually build decoder embeddings:

1. Convert transcript tokens to embeddings
2. Expand the trainable soft prompt for the batch
3. Concatenate prompt embeddings before token embeddings
4. Extend labels with `-100` so prompt positions do not contribute to loss


In [ ]:
if model is not None and processor is not None and soft_prompt is not None:
    batch_examples = examples[:1]
    audio_inputs = prepare_audio_inputs(batch_examples, processor, device, torch_dtype)
    decoder_input_ids, labels = tokenize_transcripts(batch_examples, processor, device)

    token_embeddings = model.get_decoder().embed_tokens(decoder_input_ids)
    prompt_embeddings = soft_prompt(batch_size=decoder_input_ids.shape[0])
    decoder_inputs_embeds = torch.cat([prompt_embeddings, token_embeddings], dim=1)

    prompt_label_mask = torch.full(
        (labels.shape[0], prompt_length),
        -100,
        dtype=labels.dtype,
        device=labels.device,
    )
    labels_with_prompt = torch.cat([prompt_label_mask, labels], dim=1)

    print("Prepared decoder embeddings with soft prompt:")
    print(f"  token_embeddings     : {list(token_embeddings.shape)}")
    print(f"  prompt_embeddings    : {list(prompt_embeddings.shape)}")
    print(f"  decoder_inputs_embeds: {list(decoder_inputs_embeds.shape)}")
    print(f"  labels_with_prompt   : {list(labels_with_prompt.shape)}")
else:
    print("Real decoder embedding preparation skipped.")
    print("Expected flow: token embeddings + prompt embeddings -> concatenated decoder_inputs_embeds.")


## 7. Step 4 - Run a Training Smoke Test

This one-step smoke test verifies that gradients flow into the soft prompt and not into the frozen Whisper model.

For meaningful fine-tuning, replace the mock examples with real speech/transcript pairs and train for many steps with validation.


In [ ]:
if model is not None and processor is not None and soft_prompt is not None:
    model.train()
    soft_prompt.train()

    batch_examples = examples[:1]
    audio_inputs = prepare_audio_inputs(batch_examples, processor, device, torch_dtype)
    decoder_input_ids, labels = tokenize_transcripts(batch_examples, processor, device)

    token_embeddings = model.get_decoder().embed_tokens(decoder_input_ids)
    prompt_embeddings = soft_prompt(batch_size=decoder_input_ids.shape[0])
    decoder_inputs_embeds = torch.cat([prompt_embeddings, token_embeddings], dim=1)

    prompt_label_mask = torch.full((labels.shape[0], prompt_length), -100, dtype=labels.dtype, device=labels.device)
    labels_with_prompt = torch.cat([prompt_label_mask, labels], dim=1)

    optimizer = torch.optim.AdamW(soft_prompt.parameters(), lr=1e-3)
    optimizer.zero_grad(set_to_none=True)
    outputs = model(
        input_features=audio_inputs["input_features"],
        attention_mask=audio_inputs.get("attention_mask"),
        decoder_inputs_embeds=decoder_inputs_embeds,
        labels=labels_with_prompt,
    )
    loss = outputs.loss
    loss.backward()
    optimizer.step()

    losses.append(float(loss.detach().cpu()))
    prompt_grad = soft_prompt.prompt_embeddings.grad
    print(f"Smoke-test loss       : {losses[-1]:.4f}")
    print(f"Prompt gradient norm  : {prompt_grad.norm().item():.6f}")
    print("Optimizer updated soft prompt parameters only.")
else:
    print("Skipped real training smoke test in this environment.")
    print("To run it: use a CUDA environment or set RUN_WHISPER_FINETUNE_SMOKE=1 before launching Jupyter.")


## 8. Visualize Prompt Diagnostics

This plot shows two diagnostics from the soft prompt smoke test.

The left panel compares parameter counts. The gray bar is the frozen Whisper model, and the green bar is the trainable soft prompt. The soft prompt bar is tiny because it has only `prompt_length × decoder_embedding_dim` trainable values.

The right panel shows the **smoke-test loss**. The blue dot is the loss value from the single training step we ran in Step 7. In this notebook, there is only one dot because we intentionally run only one optimizer step to prove that the training path works. If you trained for many steps on real data, this panel would show a curve with one point per step, and you would usually hope to see the loss decrease over time.

Important: this blue dot does not prove that Whisper is meaningfully fine-tuned. It only proves that the model computed a loss, gradients flowed into the soft prompt, and the optimizer updated the prompt matrix.


In [ ]:
plot_prompt_diagnostics(base_params, prompt_params, losses)
plt.show()

print(f"Frozen Whisper parameters used for plot: {base_params:,}")
print(f"Soft prompt parameters used for plot   : {prompt_params:,}")
if losses:
    print(f"Recorded smoke-test losses             : {losses}")


## 9. Flow Summary

```text
Mock audio + transcript
    -> processor(audio, text)
Whisper input_features + transcript token IDs
    -> freeze all Whisper weights
Create trainable soft prompt embeddings
    -> prepend prompt embeddings to decoder token embeddings
Forward pass computes loss
    -> optimizer updates soft prompt only
```

### Key Point

**Soft prompt tuning adapts Whisper by learning a small set of continuous decoder prompt embeddings while keeping the full large-v3-turbo model frozen.** It is extremely parameter-efficient, but usually less expressive than LoRA.


## Conclusion

### What You Learned

In this tutorial, you learned how soft prompt tuning works for Whisper large-v3-turbo.

Key concepts covered:

1. **Soft prompt theory**
   - Soft prompts are trainable continuous vectors
   - They are not human-readable text tokens
   - They steer the decoder through embeddings

2. **Whisper fine-tuning mechanics**
   - Audio becomes encoder input features
   - Transcripts become decoder token labels
   - Prompt positions are masked with `-100` in the loss

3. **Parameter-efficient training**
   - Whisper remains frozen
   - Only the soft prompt matrix is trainable
   - Gradient checks confirm the prompt receives updates

### Next Steps

You can now:

- Replace mock data with real audio/transcript pairs
- Increase prompt length and compare validation WER
- Compare soft prompt tuning against LoRA
- Save learned prompt embeddings with `torch.save(...)`
- Use the prompt as a lightweight domain adapter
